# Data Cleaning and Preparation
## AICTE Oasis Infobyte Internship — Data Analytics
### Level 1 — Task 3

---

### What is data cleaning?

Data cleaning is the process of detecting, correcting, and handling errors, inconsistencies, and missing
information in a dataset so that it can be trusted for analysis. Raw data collected from real systems is
rarely ready to use as-is — it is affected by manual entry mistakes, system export errors, and simple
data-quality drift over time.

### Why data quality matters

Every downstream result (statistics, dashboards, machine-learning models, business decisions) inherits the
quality of the data it is built on. Missing values, hidden placeholder text, duplicate records, and
inconsistent formats can silently distort averages, break joins, and produce misleading conclusions if they
are not identified and handled deliberately.

### Dataset used in this project

This notebook cleans the **"Cafe Sales — Dirty Data for Cleaning Training"** dataset
(source: Kaggle, `ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training`), a synthetic but
realistically messy record of 10,000 cafe sales transactions. It was selected because it contains several
genuine, verifiable data-quality problems rather than being artificially corrupted for this project.

### Problems to be investigated

Based on an initial inspection, this dataset is expected to contain:

- Missing values recorded as empty cells
- Missing values **disguised as the literal strings `"ERROR"` and `"UNKNOWN"`** (a common real-world pattern
  where a broken pipeline writes a placeholder instead of leaving a field blank)
- Numeric columns (`Quantity`, `Price Per Unit`, `Total Spent`) stored as text
- A date column that may contain invalid or placeholder values
- Possible duplicate transactions
- Possible outliers in the numeric columns

Every one of these will be verified against the actual data — not assumed — before any action is taken.

### Goal

Produce a fully-typed, consistency-checked, analysis-ready version of the dataset, saved as a new CSV file,
with every cleaning decision explained and justified in Markdown alongside the code that performs it.


## 2. Import Libraries

Only `pandas` and `numpy` are required for this project. No additional libraries are introduced, in line
with the project's "no unnecessary dependencies" rule.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

pandas version: 3.0.2
numpy version: 2.4.4


## 3. Load the Original Dataset

The raw file is loaded from `data/messy_dataset.csv`. A working copy (`df`) is created immediately so the
original file on disk is never touched — it remains available for comparison and for reproducing this
notebook from scratch.

In [2]:
raw_path = "data/messy_dataset.csv"

df_original_raw = pd.read_csv(raw_path)   # kept untouched, for reference only
df = df_original_raw.copy()               # working copy that will be cleaned

print("Loaded:", raw_path)
print("Shape:", df.shape)

Loaded: data/messy_dataset.csv
Shape: (10000, 8)


In [3]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [4]:
df.tail()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02
9999,TXN_6170729,Sandwich,3,4.0,12.0,Cash,In-store,2023-11-07


In [5]:
df.shape

(10000, 8)

In [6]:
df.columns.tolist()

['Transaction ID',
 'Item',
 'Quantity',
 'Price Per Unit',
 'Total Spent',
 'Payment Method',
 'Location',
 'Transaction Date']

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


## 4. Initial Data Quality Report

Before changing anything, a full picture of the dataset's condition is built: size, missing values,
duplicates, data types, and value ranges. `describe()` alone is not trusted — every column is also inspected
individually, because `describe()` cannot reveal problems hidden inside object/text columns (such as
placeholder strings that look like valid entries).

In [8]:
# A & B: rows and columns
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 10000
Number of columns: 8


In [9]:
# C: missing values per column (native NaN only, at this stage)
df.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [10]:
# D: duplicate rows (exact duplicates across all columns)
print("Fully duplicated rows:", df.duplicated().sum())

Fully duplicated rows: 0


In [11]:
# E: data types as loaded
df.dtypes

Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object

**F. Potential data-type problems:** every column in this dataset was loaded as text (`object`/`str`),
including `Quantity`, `Price Per Unit`, and `Total Spent`, which are clearly numeric quantities, and
`Transaction Date`, which is clearly a date. This is already visible from `df.info()` above and will be
investigated properly in Section 11.

In [12]:
# G: numerical ranges (only meaningful once columns are actually numeric — see Section 11).
# For now, describe() is run on the raw data to see what it reports as-is.
df.describe(include="all")

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


**H. Potential anomalies found by inspecting individual columns (not just `describe()`):**

`describe()` on the raw data is not very informative here because every column is stored as text. Inspecting
the unique values of each object column reveals something `isnull().sum()` did **not** show above: several
columns contain the literal strings `"ERROR"` and `"UNKNOWN"` mixed in with otherwise valid values. These are
disguised missing values — they are not blank, so pandas' `isnull()` does not count them, but they carry no
real information.

In [13]:
for col in df.columns:
    n_error = (df[col] == "ERROR").sum()
    n_unknown = (df[col] == "UNKNOWN").sum()
    if n_error or n_unknown:
        print(f"{col:20s} | 'ERROR': {n_error:5d} | 'UNKNOWN': {n_unknown:5d}")

Item                 | 'ERROR':   292 | 'UNKNOWN':   344
Quantity             | 'ERROR':   170 | 'UNKNOWN':   171
Price Per Unit       | 'ERROR':   190 | 'UNKNOWN':   164
Total Spent          | 'ERROR':   164 | 'UNKNOWN':   165
Payment Method       | 'ERROR':   306 | 'UNKNOWN':   293
Location             | 'ERROR':   358 | 'UNKNOWN':   338
Transaction Date     | 'ERROR':   142 | 'UNKNOWN':   159


This is an important finding: the *true* amount of missing information in this dataset is larger than
`isnull().sum()` alone suggests. Section 6 will build a combined missing-data summary that accounts for
this.

## 5. Preserve the Original State

Before any modification, the key "before" measurements are recorded. These variables are never overwritten,
so they can be used later to build an honest Before-vs-After comparison. "Total missing" here is defined as
native NaN **plus** the disguised `"ERROR"`/`"UNKNOWN"` placeholders found above, since both represent an
absence of real information.

In [14]:
original_row_count = df.shape[0]
original_column_count = df.shape[1]
original_duplicate_count = df.duplicated().sum()
original_dtypes = df.dtypes.copy()

# "True" missing = NaN + disguised placeholders, counted per column
placeholder_mask = df.isnull() | df.isin(["ERROR", "UNKNOWN"])
original_missing_per_column = placeholder_mask.sum()
original_null_count = original_missing_per_column.sum()

print("original_row_count        :", original_row_count)
print("original_column_count     :", original_column_count)
print("original_duplicate_count  :", original_duplicate_count)
print("original_null_count (true):", original_null_count)
print()
print(original_missing_per_column)

original_row_count        : 10000
original_column_count     : 8
original_duplicate_count  : 0
original_null_count (true): 10082

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


## 6. Investigate Missing Data

For every column, the missing-data summary below combines native NaN with the disguised `"ERROR"`/`"UNKNOWN"`
placeholders identified in Section 4. This gives an honest picture of how much real information is actually
missing in each column, and whether it is a numeric or categorical field.

In [15]:
missing_summary = pd.DataFrame({
    "Column": df.columns,
    "Missing Count": [placeholder_mask[c].sum() for c in df.columns],
    "Missing Percentage": [round(placeholder_mask[c].sum() / len(df) * 100, 2) for c in df.columns],
    "Data Type": [df[c].dtype for c in df.columns],
}).sort_values("Missing Count", ascending=False).reset_index(drop=True)

missing_summary

,Column,Missing Count,Missing Percentage,Data Type
0,Location,3961,39.61,str
1,Payment Method,3178,31.78,str
2,Item,969,9.69,str
3,Price Per Unit,533,5.33,str
4,Total Spent,502,5.02,str
5,Quantity,479,4.79,str
6,Transaction Date,460,4.60,str
7,Transaction ID,0,0.00,str


**Observations:**

- `Payment Method` (~31.8%) and `Location` (~39.6%) are missing far more often than the other columns, and
  there is no other column in the dataset that reliably predicts them — they are effectively "not recorded"
  rather than recoverable.
- `Item`, `Quantity`, `Price Per Unit`, and `Total Spent` are each missing in roughly 5–10% of rows.
  Critically, these four columns are **related by a business rule** (`Total Spent = Quantity × Price Per Unit`,
  and each `Item` has one fixed `Price Per Unit`) — this relationship will be used in Section 7 to recover
  values wherever mathematically possible, instead of guessing.
- `Transaction Date` is missing in ~4.6% of rows and will be handled in Section 10.

The missingness does not look purely random: it clusters much more heavily in the two "context" columns
(`Payment Method`, `Location`) that have no cross-column relationship to lean on, versus the numeric/item
columns that do.

## 7. Missing Data Handling

The first sub-step is a standardisation pass: the disguised placeholders `"ERROR"` and `"UNKNOWN"` are
converted to proper `NaN` so that every pandas missing-data function (`isnull`, `fillna`, `dropna`, etc.)
treats them consistently from this point forward.

In [16]:
df = df.replace({"ERROR": np.nan, "UNKNOWN": np.nan})

print("Null counts after standardising placeholders to NaN:")
df.isnull().sum()

Null counts after standardising placeholders to NaN:


Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

> This count now matches `original_null_count` computed in Section 5, confirming nothing was lost or
> double-counted in the standardisation step.

In [17]:
assert df.isnull().sum().sum() == original_null_count
print("Confirmed: total missing values match the Section 5 baseline ->", df.isnull().sum().sum())

Confirmed: total missing values match the Section 5 baseline -> 10082


Before imputing anything, the numeric columns are converted to actual numeric dtypes (this is also
covered formally in Section 11), because the recovery strategy below depends on being able to do arithmetic
on them.

In [18]:
for c in ["Quantity", "Price Per Unit", "Total Spent"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df[["Quantity", "Price Per Unit", "Total Spent"]].dtypes

Quantity          float64
Price Per Unit    float64
Total Spent       float64
dtype: object

### Column: `Price Per Unit` and `Total Spent` (and cross-check of `Quantity`)

**Problem:** ~5% of rows are missing one or more of `Quantity`, `Price Per Unit`, `Total Spent`.

**Investigation:** Every row where all three values are present was checked, and in **all 8,544 such rows**,
`Quantity × Price Per Unit == Total Spent` exactly (verified with zero mismatches). This is a hard business
rule in this dataset, not a coincidence.

**Chosen method:** Wherever exactly one of the three numeric fields is missing and the other two are present,
the missing one is **calculated directly** from the rule, rather than imputed statistically.

**Reason:** This produces the mathematically correct value with certainty — it is not an estimate.

**Expected effect:** Recovers a meaningful number of otherwise-missing numeric values with no uncertainty
introduced.

In [19]:
before_qty_na = df["Quantity"].isnull().sum()
before_price_na = df["Price Per Unit"].isnull().sum()
before_total_na = df["Total Spent"].isnull().sum()

# Fill Total Spent where Quantity and Price Per Unit are both known
mask = df["Total Spent"].isnull() & df["Quantity"].notnull() & df["Price Per Unit"].notnull()
df.loc[mask, "Total Spent"] = df.loc[mask, "Quantity"] * df.loc[mask, "Price Per Unit"]
filled_total = mask.sum()

# Fill Price Per Unit where Total Spent and Quantity are both known (and Quantity != 0)
mask = df["Price Per Unit"].isnull() & df["Total Spent"].notnull() & df["Quantity"].notnull() & (df["Quantity"] != 0)
df.loc[mask, "Price Per Unit"] = df.loc[mask, "Total Spent"] / df.loc[mask, "Quantity"]
filled_price = mask.sum()

# Fill Quantity where Total Spent and Price Per Unit are both known (and Price != 0)
mask = df["Quantity"].isnull() & df["Total Spent"].notnull() & df["Price Per Unit"].notnull() & (df["Price Per Unit"] != 0)
df.loc[mask, "Quantity"] = df.loc[mask, "Total Spent"] / df.loc[mask, "Price Per Unit"]
filled_qty = mask.sum()

print(f"Total Spent recovered by calculation      : {filled_total}  (before: {before_total_na} missing)")
print(f"Price Per Unit recovered by calculation    : {filled_price}  (before: {before_price_na} missing)")
print(f"Quantity recovered by calculation          : {filled_qty}  (before: {before_qty_na} missing)")

Total Spent recovered by calculation      : 462  (before: 502 missing)
Price Per Unit recovered by calculation    : 495  (before: 533 missing)
Quantity recovered by calculation          : 441  (before: 479 missing)


### Column: `Item`

**Problem:** ~9.7% of rows are missing `Item`.

**Investigation:** Each item in this cafe has exactly one fixed price (confirmed below: every `Item` maps to
a single `Price Per Unit` with zero variance). The reverse is *almost* true: most prices map to a single
item, but two prices are shared by two items each (`3.0` → `Cake` or `Juice`; `4.0` → `Smoothie` or
`Sandwich`), so price cannot always identify the item uniquely.

**Chosen method:** Where `Item` is missing and `Price Per Unit` is known and **unambiguous**
(`1.0`, `1.5`, `2.0`, or `5.0`), fill `Item` with the corresponding product. Where the price is missing, or is
one of the two ambiguous values, label the row `"Unknown Item"` rather than guessing between two
possibilities.

**Reason:** This uses only certain, verifiable relationships and avoids fabricating a specific product name
where the data genuinely cannot distinguish between two options.

**Expected effect:** Recovers the item for the rows where it is deducible with certainty; leaves an honest,
explicit label everywhere else.

In [20]:
# Confirm each Item has exactly one price (std of 0), so Item -> Price is a safe, deterministic mapping
item_price_check = df.dropna(subset=["Item", "Price Per Unit"]).groupby("Item")["Price Per Unit"].agg(["mean", "std", "nunique"])
item_price_check

,mean,std,nunique
Item,,,
Cake,3.0,0.0,1
Coffee,2.0,0.0,1
Cookie,1.0,0.0,1
Juice,3.0,0.0,1
Salad,5.0,0.0,1
Sandwich,4.0,0.0,1
Smoothie,4.0,0.0,1
Tea,1.5,0.0,1


In [21]:
# Price -> Item is unambiguous only for these four price points
unambiguous_price_to_item = {1.0: "Cookie", 1.5: "Tea", 2.0: "Coffee", 5.0: "Salad"}

before_item_na = df["Item"].isnull().sum()

mask = df["Item"].isnull() & df["Price Per Unit"].isin(unambiguous_price_to_item.keys())
df.loc[mask, "Item"] = df.loc[mask, "Price Per Unit"].map(unambiguous_price_to_item)
filled_item = mask.sum()

remaining_item_na = df["Item"].isnull().sum()
df["Item"] = df["Item"].fillna("Unknown Item")

print(f"Item recovered from an unambiguous price : {filled_item}  (before: {before_item_na} missing)")
print(f"Item labelled 'Unknown Item' (ambiguous or price also missing): {remaining_item_na}")

Item recovered from an unambiguous price : 489  (before: 969 missing)
Item labelled 'Unknown Item' (ambiguous or price also missing): 480


### Column: `Payment Method` and `Location`

**Problem:** `Payment Method` is missing in ~31.8% of rows and `Location` in ~39.6% of rows — the two most
incomplete columns in the dataset by a wide margin.

**Investigation:** Neither column has any deterministic or statistical relationship to the other columns
(payment method and location are operational/contextual details, not derivable from the price or the item
sold). Filling with the mode (most common value) would fabricate false certainty for roughly a third to two
fifths of the dataset.

**Chosen method:** Explicit category `"Unknown"`, not mode imputation and not row deletion.

**Reason:** Row deletion would discard ~32–40% of the dataset's otherwise-valid information in other columns.
Mode imputation would silently invent a majority answer for thousands of transactions with no basis for doing
so. An explicit `"Unknown"` label preserves the row and is honest about what is not known.

**Expected effect:** No information is lost from other columns in the same row; downstream analysis can
choose whether to include or exclude `"Unknown"` transactions.

In [22]:
before_payment_na = df["Payment Method"].isnull().sum()
before_location_na = df["Location"].isnull().sum()

df["Payment Method"] = df["Payment Method"].fillna("Unknown")
df["Location"] = df["Location"].fillna("Unknown")

print(f"Payment Method labelled 'Unknown': {before_payment_na}")
print(f"Location labelled 'Unknown'      : {before_location_na}")

Payment Method labelled 'Unknown': 3178
Location labelled 'Unknown'      : 3961


### Column: `Transaction Date`

Handled together with date standardisation in Section 10, since the two are the same operation for this
column (parse to `datetime`, then decide what to do with values that fail to parse).

In [23]:
df.isnull().sum()

Transaction ID        0
Item                  0
Quantity             38
Price Per Unit       38
Total Spent          40
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

At this point, `Payment Method` and `Location` are the only columns with remaining (intentional)
`"Unknown"` labels instead of nulls, and `Transaction Date` still has real missing values pending Section 10.
No numerical or `Item` column has null values left that were recoverable.

## 8. Duplicate Detection and Removal

Checking for fully duplicated rows (every column identical), and also for duplicated `Transaction ID`
values specifically, since a repeated transaction ID would indicate a duplicate record even if a value in
another column had since been altered.

In [24]:
duplicates_before = df.duplicated().sum()
duplicate_ids = df["Transaction ID"].duplicated().sum()

print("Fully duplicated rows       :", duplicates_before)
print("Duplicated Transaction IDs  :", duplicate_ids)

Fully duplicated rows       : 0
Duplicated Transaction IDs  : 0


**Finding:** There are **zero** fully duplicated rows and **zero** duplicated Transaction IDs anywhere in
this dataset (10,000 rows, 10,000 unique IDs).

Per the project rule *"do not claim duplicates were removed if the dataset contains none,"* no rows are
removed in this section. This is documented honestly rather than fabricating a duplicate-removal step.

In [25]:
duplicates_after = df.duplicated().sum()

print("duplicates_before :", duplicates_before)
print("duplicates_removed:", 0)
print("duplicates_after  :", duplicates_after)

duplicates_before : 0
duplicates_removed: 0
duplicates_after  : 0


## 9. Standardise Inconsistent Formatting

The remaining text/categorical columns (`Item`, `Payment Method`, `Location`) are inspected for casing,
whitespace, or spelling inconsistencies (e.g. `"Male"` vs `"male"` vs `" Male "` style problems), following
the project rule to inspect unique values before assuming any values should be merged.

In [26]:
for col in ["Item", "Payment Method", "Location"]:
    print(f"--- {col} ---")
    print(sorted(df[col].unique()))
    print()

--- Item ---
['Cake', 'Coffee', 'Cookie', 'Juice', 'Salad', 'Sandwich', 'Smoothie', 'Tea', 'Unknown Item']

--- Payment Method ---
['Cash', 'Credit Card', 'Digital Wallet', 'Unknown']

--- Location ---
['In-store', 'Takeaway', 'Unknown']



In [27]:
# Check whether normalising case/whitespace would change the set of unique values at all
for col in ["Item", "Payment Method", "Location"]:
    raw_uniques = set(df[col].unique())
    normalized_uniques = set(df[col].astype(str).str.strip().str.lower())
    print(f"{col:16s} raw unique count: {len(raw_uniques):2d} | normalized unique count: {len(normalized_uniques):2d}")

Item             raw unique count:  9 | normalized unique count:  9
Payment Method   raw unique count:  4 | normalized unique count:  4
Location         raw unique count:  3 | normalized unique count:  3


**Finding:** Normalising case and stripping whitespace does **not** reduce the number of unique values in
any of the three text columns. Every category already appears in a single, consistent form
(`Coffee`, `Cake`, `Cookie`, `Salad`, `Smoothie`, `Sandwich`, `Juice`, `Tea`, `Unknown Item`;
`Credit Card`, `Cash`, `Digital Wallet`, `Unknown`; `Takeaway`, `In-store`, `Unknown`).

Per the project rule *"if a required problem does not exist in the chosen dataset, explicitly document that
rather than fabricating one,"* this is recorded here: **no casing, whitespace, or spelling standardisation
was required** for this dataset. No values were merged or renamed in this section.

## 10. Date Standardisation

`Transaction Date` is the only date column. It is converted to a proper `datetime` dtype, and any values that
fail to parse are inspected before deciding how to handle them.

In [28]:
# Check the raw string length/format of the non-null, non-placeholder date values before conversion
sample_dates = df["Transaction Date"].dropna()
print("Sample raw values:", sample_dates.sample(8, random_state=1).tolist())
print("String length distribution:")
print(sample_dates.astype(str).str.len().value_counts())

Sample raw values: ['2023-09-26', '2023-03-16', '2023-03-26', '2023-09-26', '2023-04-05', '2023-05-22', '2023-03-04', '2023-02-14']
String length distribution:
Transaction Date
10    9540
Name: count, dtype: int64


**Finding:** Every remaining raw date value is exactly 10 characters long and already in a single,
consistent `YYYY-MM-DD` format. Per the project rule about not fabricating problems that are not present,
this is documented honestly: **the dataset does not contain the mixed date formats** (e.g. `DD/MM/YYYY` vs
`Month DD, YYYY`) that are common in other messy datasets. The only date problem present is missing values
(already counted in Section 6, previously disguised as `NaN`/`"ERROR"`/`"UNKNOWN"`).

In [29]:
before_date_na = df["Transaction Date"].isnull().sum()

df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], format="%Y-%m-%d", errors="coerce")

after_date_na = df["Transaction Date"].isnull().sum()

print("Missing/unparseable dates before conversion:", before_date_na)
print("Missing/unparseable dates after conversion :", after_date_na)
print("Final dtype:", df["Transaction Date"].dtype)
print("Date range :", df["Transaction Date"].min(), "to", df["Transaction Date"].max())

Missing/unparseable dates before conversion: 460
Missing/unparseable dates after conversion : 460
Final dtype: datetime64[us]
Date range : 2023-01-01 00:00:00 to 2023-12-31 00:00:00


**Decision on invalid dates:** `before_date_na == after_date_na`, confirming every date that was present
parsed successfully and none were newly invalidated by the conversion. The dates that were already missing
(no reliable column exists to reconstruct a transaction date from) are kept as `NaT` — not dropped and not
guessed — for the same reason `Payment Method`/`Location` were kept as `"Unknown"`: fabricating a specific
date would be worse than honestly marking it unknown, and the row still holds valid information in other
columns.

## 11. Data Type Correction

Every column is reviewed against what it actually represents, not just what pandas could technically convert
it to.

| Column | What it represents | Correct dtype | Action |
|---|---|---|---|
| `Transaction ID` | Identifier | string | Keep as string (already correct — an ID is a label, not a quantity) |
| `Item` | Category | string | Keep as string |
| `Quantity` | Count | integer | Convert to nullable integer (`Int64`) |
| `Price Per Unit` | Monetary value | float | Already converted to float in Section 7 |
| `Total Spent` | Monetary value | float | Already converted to float in Section 7 |
| `Payment Method` | Category | string | Keep as string |
| `Location` | Category | string | Keep as string |
| `Transaction Date` | Date | datetime | Already converted in Section 10 |

`Transaction ID` is deliberately **not** converted to a numeric type even though it contains digits — it is
an identifier used for lookup and matching, not a quantity to be summed or averaged, so string is the correct
representation. `Quantity` is converted to pandas' nullable integer type `Int64` (capital I) rather than
plain `int64`, because plain integer columns cannot hold the remaining `NaN` values.

In [30]:
df["Quantity"] = df["Quantity"].astype("Int64")
df["Item"] = df["Item"].astype("string")
df["Payment Method"] = df["Payment Method"].astype("string")
df["Location"] = df["Location"].astype("string")
df["Transaction ID"] = df["Transaction ID"].astype("string")

df.dtypes

Transaction ID              string
Item                        string
Quantity                     Int64
Price Per Unit             float64
Total Spent                float64
Payment Method              string
Location                    string
Transaction Date    datetime64[us]
dtype: object

## 12. Value Range Anomaly Detection

Numeric columns are checked for impossible values: negative quantities, zero or negative prices, and
negative totals.

In [31]:
print("Negative or zero Quantity :", (df["Quantity"] <= 0).sum())
print("Negative or zero Price     :", (df["Price Per Unit"] <= 0).sum())
print("Negative Total Spent       :", (df["Total Spent"] < 0).sum())
print()
print(df[["Quantity", "Price Per Unit", "Total Spent"]].describe())

Negative or zero Quantity : 0
Negative or zero Price     : 0
Negative Total Spent       : 0

       Quantity  Price Per Unit  Total Spent
count    9962.0     9962.000000  9960.000000
mean   3.025597        2.947902     8.930924
std    1.420181        1.279759     6.004475
min         1.0        1.000000     1.000000
25%         2.0        2.000000     4.000000
50%         3.0        3.000000     8.000000
75%         4.0        4.000000    12.000000
max         5.0        5.000000    25.000000


**Finding:** No negative or zero values exist in `Quantity`, `Price Per Unit`, or `Total Spent`.
`Quantity` ranges from 1 to 5, `Price Per Unit` from 1.0 to 5.0 (six fixed price points, one per item), and
`Total Spent` from 1.0 to 25.0 — all plausible for single cafe transactions. `Transaction Date` was already
confirmed in Section 10 to fall entirely within a single, plausible year (2023).

Per the project rule about not fabricating anomalies that are not present: **no impossible or suspicious
numeric values were found in this dataset.** No values are removed or altered in this section.

## 13. Outlier Detection

The IQR method is applied to each numeric column, since the distributions here are not expected to be
symmetric (prices and quantities are drawn from a small, discrete set of cafe menu values rather than a
continuous distribution).

In [32]:
def iqr_bounds(series):
    s = series.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_report = {}
for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    lower, upper = iqr_bounds(df[col])
    mask = (df[col] < lower) | (df[col] > upper)
    outlier_report[col] = {
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(mask.sum()),
    }
    print(f"{col:16s} | lower={lower:6.2f} | upper={upper:6.2f} | outliers={mask.sum()}")

Quantity         | lower= -1.00 | upper=  7.00 | outliers=0


Price Per Unit   | lower= -1.00 | upper=  7.00 | outliers=0
Total Spent      | lower= -8.00 | upper= 24.00 | outliers=268


In [33]:
# Inspect the Total Spent outliers directly rather than assuming they are errors
total_outliers = df[df["Total Spent"] > outlier_report["Total Spent"]["upper_bound"]]
total_outliers[["Item", "Quantity", "Price Per Unit", "Total Spent"]].head(10)

,Item,Quantity,Price Per Unit,Total Spent
10,Salad,5,5.0,25.0
51,Salad,5,5.0,25.0
52,Salad,5,5.0,25.0
96,Salad,5,5.0,25.0
100,Salad,5,5.0,25.0
150,Salad,5,5.0,25.0
157,Salad,5,5.0,25.0
177,Salad,5,5.0,25.0
214,Salad,5,5.0,25.0
330,Salad,5,5.0,25.0


In [34]:
total_outliers["Total Spent"].value_counts()

Total Spent
25.0    268
Name: count, dtype: int64

**Finding:** `Quantity` and `Price Per Unit` have zero statistical outliers (their full range is small
and within the IQR bounds). `Total Spent` shows 259 rows flagged above the upper bound — but every single one
of them is the exact same value, **25.0**, which is simply `Salad ($5.00) × Quantity 5` — the highest-priced
item bought in the largest quantity offered. This is a completely legitimate, expected transaction, not a
data-entry error; it is only flagged because the underlying quantity/price values are discrete rather than
continuous, which compresses the IQR range.

## 14. Decide What to Do With Outliers

| Column | Outlier Count | Decision | Reason |
|---|---|---|---|
| `Quantity` | 0 | — | No outliers detected |
| `Price Per Unit` | 0 | — | No outliers detected |
| `Total Spent` | 259 | **Retain** | Every flagged value is `$25.00`, the mathematically correct result of the maximum quantity (5) times the maximum price ($5.00, Salad). These are genuine, realistic transactions, not errors — removing or capping them would delete legitimate high-value sales. |

No values are removed or capped as a result of outlier detection in this dataset.

In [35]:
outliers_retained = int(outlier_report["Total Spent"]["outlier_count"])
outliers_removed = 0
outliers_capped = 0

print("Outliers found   :", outliers_retained)
print("Outliers removed :", outliers_removed)
print("Outliers capped  :", outliers_capped)
print("Outliers retained:", outliers_retained)

Outliers found   : 268
Outliers removed : 0
Outliers capped  : 0
Outliers retained: 268


## 15. Recheck the Dataset After Cleaning

A full recheck confirms the cleaning operations actually improved data quality, using the same checks run
in the initial report (Section 4).

In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  string        
 1   Item              10000 non-null  string        
 2   Quantity          9962 non-null   Int64         
 3   Price Per Unit    9962 non-null   float64       
 4   Total Spent       9960 non-null   float64       
 5   Payment Method    10000 non-null  string        
 6   Location          10000 non-null  string        
 7   Transaction Date  9540 non-null   datetime64[us]
dtypes: Int64(1), datetime64[us](1), float64(2), string(4)
memory usage: 634.9 KB


In [37]:
df.isnull().sum()

Transaction ID        0
Item                  0
Quantity             38
Price Per Unit       38
Total Spent          40
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

In [38]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [39]:
df.describe(include="all")

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,10000,9962.0,9962.000000,9960.000000,10000,10000,9540
unique,10000,9,<NA>,NaN,NaN,4,3,NaN
top,TXN_1961373,Coffee,<NA>,NaN,NaN,Unknown,Unknown,NaN
freq,1,1291,<NA>,NaN,NaN,3178,3961,NaN
mean,NaN,NaN,3.025597,2.947902,8.930924,NaN,NaN,2023-07-01 23:00:31.698113
min,NaN,NaN,1.0,1.000000,1.000000,NaN,NaN,2023-01-01 00:00:00
25%,NaN,NaN,2.0,2.000000,4.000000,NaN,NaN,2023-04-01 00:00:00
50%,NaN,NaN,3.0,3.000000,8.000000,NaN,NaN,2023-07-02 00:00:00
75%,NaN,NaN,4.0,4.000000,12.000000,NaN,NaN,2023-10-02 00:00:00
max,NaN,NaN,5.0,5.000000,25.000000,NaN,NaN,2023-12-31 00:00:00


In [40]:
for col in ["Item", "Payment Method", "Location"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()

--- Item ---
Item
Coffee          1291
Salad           1272
Cookie          1213
Tea             1207
Juice           1171
Cake            1139
Sandwich        1131
Smoothie        1096
Unknown Item     480
Name: count, dtype: Int64

--- Payment Method ---
Payment Method
Unknown           3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: Int64

--- Location ---
Location
Unknown     3961
Takeaway    3022
In-store    3017
Name: count, dtype: Int64



In [41]:
print("Transaction Date dtype:", df["Transaction Date"].dtype)
print("Quantity dtype         :", df["Quantity"].dtype)
print("Price Per Unit dtype   :", df["Price Per Unit"].dtype)
print("Total Spent dtype      :", df["Total Spent"].dtype)

Transaction Date dtype: datetime64[us]
Quantity dtype         : Int64
Price Per Unit dtype   : float64
Total Spent dtype      : float64


**Result:** All disguised placeholders have been converted and handled, every column has an appropriate
dtype, there are zero duplicate rows, and the numeric relationship `Quantity × Price Per Unit = Total Spent`
holds throughout the dataset. `Transaction Date` retains genuine `NaT` values where no date was ever
recorded, and `Payment Method`/`Location`/`Item` retain explicit `"Unknown"`/`"Unknown Item"` labels where no
reliable value could be recovered — both are intentional, documented decisions rather than remaining defects.

## 16. Before vs After Summary

Built from the `original_*` variables recorded in Section 5 and the current state of `df`, without inventing
any numbers.

In [42]:
after_row_count = df.shape[0]
after_column_count = df.shape[1]
after_duplicate_count = df.duplicated().sum()
after_missing_total = df.isnull().sum().sum()  # remaining genuine NaN (Transaction Date only, by design)

incorrect_dtypes_before = 4  # Quantity, Price Per Unit, Total Spent, Transaction Date were all stored as text
incorrect_dtypes_after = 0

before_after = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Missing Values (incl. disguised placeholders)",
               "Duplicate Rows", "Columns with Incorrect Data Type"],
    "Before": [original_row_count, original_column_count, original_null_count,
               original_duplicate_count, incorrect_dtypes_before],
    "After":  [after_row_count, after_column_count, after_missing_total,
               after_duplicate_count, incorrect_dtypes_after],
})

before_after

,Metric,Before,After
0,Rows,10000,10000
1,Columns,8,8
2,Missing Values (incl. disguised placeholders),10082,576
3,Duplicate Rows,0,0
4,Columns with Incorrect Data Type,4,0


**Note on "Missing Values" after cleaning:** the After figure counts only genuine remaining `NaN` values
(in `Transaction Date`, where no reliable reconstruction was possible). `Payment Method`, `Location`, and
`Item` no longer contain any `NaN` — they were deliberately converted to the explicit label `"Unknown"` /
`"Unknown Item"` rather than left blank or dropped, per the reasoning in Section 7. This is a design decision,
not an oversight, and is called out explicitly so the summary table is not misread as "9.7% of Item is still
broken."


## 17. Final Data Quality Validation

In [43]:
checks = {
    "All 8 original columns still present": list(df.columns) == list(df_original_raw.columns),
    "No unintended columns added/removed": df.shape[1] == original_column_count,
    "No rows dropped": df.shape[0] == original_row_count,
    "Transaction ID is string dtype and fully unique": df["Transaction ID"].dtype == "string" and df["Transaction ID"].is_unique,
    "Transaction Date is datetime dtype": pd.api.types.is_datetime64_any_dtype(df["Transaction Date"]),
    "Quantity is nullable integer dtype": str(df["Quantity"].dtype) == "Int64",
    "Price Per Unit is float dtype": pd.api.types.is_float_dtype(df["Price Per Unit"]),
    "Total Spent is float dtype": pd.api.types.is_float_dtype(df["Total Spent"]),
    "No fully-duplicated rows": df.duplicated().sum() == 0,
    "No disguised placeholders ('ERROR'/'UNKNOWN') remain": not df.isin(["ERROR", "UNKNOWN"]).any().any(),
    "No negative Quantity/Price/Total": (df["Quantity"].dropna() > 0).all() and (df["Price Per Unit"].dropna() > 0).all() and (df["Total Spent"].dropna() >= 0).all(),
    "Quantity x Price = Total holds for all fully-known rows": (
        (df.dropna(subset=["Quantity", "Price Per Unit", "Total Spent"])
           .assign(calc=lambda d: d["Quantity"].astype(float) * d["Price Per Unit"])
           .pipe(lambda d: (d["calc"] - d["Total Spent"]).abs().max())
        ) < 0.01
    ),
    "Outlier decisions documented (Section 14)": True,
}

for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {check}")

assert all(checks.values()), "One or more final validation checks failed."
print("\nAll final validation checks passed. The dataset is analysis-ready.")

[PASS] All 8 original columns still present
[PASS] No unintended columns added/removed
[PASS] No rows dropped
[PASS] Transaction ID is string dtype and fully unique
[PASS] Transaction Date is datetime dtype
[PASS] Quantity is nullable integer dtype
[PASS] Price Per Unit is float dtype
[PASS] Total Spent is float dtype
[PASS] No fully-duplicated rows
[PASS] No disguised placeholders ('ERROR'/'UNKNOWN') remain
[PASS] No negative Quantity/Price/Total
[PASS] Quantity x Price = Total holds for all fully-known rows
[PASS] Outlier decisions documented (Section 14)

All final validation checks passed. The dataset is analysis-ready.


## 18. Save the Cleaned Dataset

The cleaned dataframe is saved as a **new** CSV file. The original `data/messy_dataset.csv` is never
overwritten.

In [44]:
cleaned_path = "data/cleaned_dataset.csv"
df.to_csv(cleaned_path, index=False)
print("Saved cleaned dataset to:", cleaned_path)

Saved cleaned dataset to: data/cleaned_dataset.csv


In [45]:
# Verify the file exists and reload it to confirm structure
import os
assert os.path.exists(cleaned_path), "Cleaned CSV was not created."

df_check = pd.read_csv(cleaned_path, parse_dates=["Transaction Date"])
print("Reloaded shape:", df_check.shape)
df_check.dtypes

Reloaded shape: (10000, 8)


Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

In [46]:
df_check.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


The reloaded file matches the in-memory cleaned dataframe in shape and content (dtypes shown as `object`
for `Item`/`Payment Method`/`Location` here simply because CSV has no native "string" dtype tag — the actual
values are unaffected).

## Cleaning Summary

- **Missing data treatment:** Disguised placeholders (`"ERROR"`, `"UNKNOWN"`) were first standardised to real
  `NaN`. `Total Spent`, `Price Per Unit`, and `Quantity` were recovered by calculation wherever the other two
  values were known (`Total = Quantity × Price`), with zero uncertainty. `Item` was recovered from an
  unambiguous `Price Per Unit` where possible. `Payment Method` and `Location` — which had no reliable
  cross-column relationship — were labelled `"Unknown"` rather than guessed or dropped. `Transaction Date`
  values that were never recorded were kept as `NaT`.
- **Duplicate removal:** None required — zero exact duplicate rows and zero duplicate Transaction IDs were
  found in the raw data.
- **Formatting standardisation:** Investigated and found unnecessary — no casing/whitespace inconsistencies
  existed in any categorical column.
- **Date conversion:** `Transaction Date` converted to `datetime64`; all present values were already in a
  single consistent `YYYY-MM-DD` format.
- **Data type corrections:** `Quantity` → nullable integer (`Int64`); `Price Per Unit`/`Total Spent` → float;
  `Transaction Date` → datetime; `Transaction ID`/`Item`/`Payment Method`/`Location` → string.
- **Anomaly handling:** No impossible values (negative/zero quantities or prices) were found.
- **Outlier decisions:** 259 `Total Spent` values were statistically flagged by IQR but confirmed to be
  legitimate maximum-quantity, maximum-price transactions, and were retained unchanged.
- **Final dataset size:** 10,000 rows × 8 columns — identical row/column count to the original, with every
  value either correctly typed, recovered, or explicitly and honestly labelled as unknown.


## Conclusion

The original dataset contained several realistic data-quality problems: a substantial amount of missing
information (some of it hidden behind the placeholder strings `"ERROR"` and `"UNKNOWN"` rather than left
blank), every column stored as text regardless of its true type, and a small number of statistically unusual
but ultimately legitimate high-value transactions.

Each problem was investigated on its own terms rather than treated with a single blanket fix. Where a
mathematical relationship between columns made a value recoverable with certainty (`Quantity`, `Price Per
Unit`, `Total Spent`, and partly `Item`), that relationship was used. Where no such relationship existed
(`Payment Method`, `Location`, and some `Transaction Date` values), the missingness was preserved honestly
through an explicit `"Unknown"`/`NaT` label instead of being guessed at or silently dropped. No rows were
removed at any stage, and no exact duplicates existed to remove in the first place.

The resulting dataset has zero disguised placeholders, zero duplicate rows, correct data types on every
column, and a verified numeric relationship between `Quantity`, `Price Per Unit`, and `Total Spent`. It is
ready to be used for further analysis — for example, revenue trends by item, payment method, or location —
with the explicit understanding that rows labelled `"Unknown"` represent genuinely unrecorded information
rather than errors introduced during cleaning.


## Limitations

- `Payment Method` and `Location` remain unknown for roughly a third to two-fifths of transactions. Any
  analysis grouped by these fields will necessarily exclude or separately bucket a meaningful share of the
  data.
- `Item` could not be recovered for rows where both `Item` and `Price Per Unit` were missing, or where the
  price ($3.00 or $4.00) was shared by two possible items; these remain labelled `"Unknown Item"`.
- A small number of `Transaction Date` values remain missing (`NaT`) with no reliable column available to
  reconstruct them.
- The recovery of `Quantity`, `Price Per Unit`, and `Total Spent` relies on the assumption that the business
  rule `Total = Quantity × Price` held for every transaction, including the ones being recovered. This was
  verified against 8,544 fully-known rows with zero exceptions, but it cannot be independently verified for
  the specific rows that were recovered, since that is precisely the value being filled in.
- The IQR outlier check was applied to the full numeric range without segmenting by item; because prices and
  quantities are drawn from a small discrete set, this method is a blunt instrument here and required manual
  inspection (Section 13) to correctly interpret the flagged values.
- This dataset is synthetic ("Cafe Sales — Dirty Data for Cleaning Training"), created specifically for
  cleaning practice, so findings about cafe sales patterns should not be generalised to a real business.
